# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook provides a step-by-step guide for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source

Dataset Croissant schema URL:
- [https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json)

In [ ]:
# Install mlcroissant if not already present
!pip install mlcroissant

## 1. Data Loading

Load dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"  # FAIR^2 dataset schema

# Load dataset using mlcroissant
dataset = mlc.Dataset(croissant_url)

# Access metadata (as an object)
meta = dataset.metadata

# Print basic dataset information
print(f"Dataset Name: {meta.name}")
print(f"Dataset Description: {meta.description}")
print(f"Dataset Identifier: {meta.identifier}")
print(f"Published: {meta.datePublished}")

## 2. Data Overview

Review available record sets and their fields. All entities are referenced by their `@id` as per Croissant schema.

We'll enumerate all `recordSet` and show their corresponding fields by `@id`.

In [ ]:
# Get recordSets from dataset metadata
record_sets = dataset.metadata.recordSets

print(f"Found {len(record_sets)} record sets:")
for rs in record_sets:
    print(f"- RecordSet name: {getattr(rs, 'name', '<no_name>')} (id: {rs['@id'] if '@id' in rs else getattr(rs, '@id', '<no_id>')})")
    # List fields
    fields = getattr(rs, 'fields', [])
    print(f"  Fields:")
    for field in fields:
        print(f"    - {getattr(field, 'name', '<no_name>')} (@id: {field['@id'] if '@id' in field else getattr(field, '@id', '<no_id>')})")

## 3. Data Extraction

Load all record sets into DataFrames for exploration.

Below we provide a structure for extracting records from each available record set by their `@id`. Fields are always referenced by their `@id`.

In [ ]:
# Prepare a list of recordSet @id's
record_set_ids = [rs['@id'] if '@id' in rs else getattr(rs, '@id', '<no_id>') for rs in dataset.metadata.recordSets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Record set {record_set_id}: {len(df)} records, columns: {df.columns.tolist()}")

# Display first record set head
if record_set_ids:
    sample_id = record_set_ids[0]
    print(f"Sample from record set {sample_id}:")
    display(dataframes[sample_id].head())

## 4. Exploratory Data Analysis (EDA)

Apply EDA steps: filter, normalize, and group using fields referenced by their `@id`.

In [ ]:
# Choose a record set and numeric field for EDA
# We'll select the first record set and look for the first numeric field
selected_rs_id = record_set_ids[0]  # Example: the first available record set
df = dataframes[selected_rs_id]

# Identify numeric fields by checking field dataType
fields = dataset.metadata.recordSets[0].fields
numeric_field_id = None
for field in fields:
    # Choose field marked as Integer or Float
    dtype = getattr(field, 'dataType', None)
    if dtype in ['schema:Integer', 'schema:Float', 'Integer', 'Float']:
        numeric_field_id = field['@id'] if '@id' in field else getattr(field, '@id', None)
        break

if numeric_field_id is None:
    print("No numeric field found for EDA.")
else:
    # Find the actual DataFrame column for this @id
    numeric_field_col = numeric_field_id
    if numeric_field_col not in df.columns:
        # Try mapping field name
        numeric_field_col = getattr(fields[0], 'name', None)

    # Apply filtering and normalization
    threshold = 10
    if numeric_field_col in df.columns:
        filtered_df = df[df[numeric_field_col] > threshold]
        print(f"Filtered records with {numeric_field_col} > {threshold}:")
        print(filtered_df.head())

        # Normalize numeric field
        filtered_df[f"{numeric_field_col}_normalized"] = (filtered_df[numeric_field_col] - filtered_df[numeric_field_col].mean()) / filtered_df[numeric_field_col].std()
        print(f"Normalized {numeric_field_col} for filtered records:")
        print(filtered_df[[numeric_field_col, f"{numeric_field_col}_normalized"]].head())

        # Grouping by another field
        group_field = None
        for field in fields:
            if field['@id'] != numeric_field_id:
                group_field = field['@id'] if '@id' in field else getattr(field, '@id', None)
                if group_field in df.columns:
                    break
        if group_field is not None:
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"Grouped data by {group_field}:")
            print(grouped_df.head())
        else:
            print("No suitable categorical/group field found.")
    else:
        print(f"Numeric field column '{numeric_field_col}' not found in DataFrame.")

## 5. Visualization

Visualize the distribution of the numeric field and relationship with a grouping field where possible.

We use matplotlib for basic plots.

In [ ]:
import matplotlib.pyplot as plt

if numeric_field_id and numeric_field_col in df.columns:
    plt.figure(figsize=(8,4))
    df[numeric_field_col].hist(bins=15)
    plt.title(f"Distribution of {numeric_field_col}")
    plt.xlabel(numeric_field_col)
    plt.ylabel('Count')
    plt.show()
    
    if group_field is not None:
        plt.figure(figsize=(10,5))
        df.groupby(group_field)[numeric_field_col].mean().plot(kind='bar')
        plt.title(f"Mean {numeric_field_col} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(f"Mean {numeric_field_col}")
        plt.show()

## 6. Conclusion

This notebook demonstrated:
- Loading metadata and record sets from the FAIR^2 clinicopathological dataset using `mlcroissant`
- Data overview with all resources and field IDs
- Extraction of all record sets into pandas DataFrames with fields referenced solely by their `@id`
- Exploratory and preprocessing steps on numeric fields, grouped and visualized by key clinical variables

Further investigation can build upon this foundation for biomarker/statistical modeling, or FAIR-compliant downstream analysis.